# GenHMM1d — R / Python parity check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamadouyamar/GenHMM1d/blob/master/parity.ipynb)

For each model class shared by the two packages, the flow is the same:

1. **R cell** — simulate a series with the CRAN
   [R GenHMM1d](https://cran.r-project.org/package=GenHMM1d) and estimate it
   with the R `EstHMMGen`;
2. **Python cell** — estimate the **same series** with the Python `EstHMMGen`
   (this repository) and print truth, R estimate, Python estimate, and the
   largest R-Python gap.

Both packages use the same EM with the same deterministic initialization, and
the settings are matched (`eps = 1e-6`, `max_iter = 10000`, minimum 100 EM
iterations), so the estimates should agree to numerical tolerance; residual
differences come only from the Nelder-Mead internals of `stats::optim` vs
`scipy.optimize.minimize`. Models: Gaussian, Poisson, zero-inflated Gaussian,
zero-inflated Poisson. (The autoregressive models and the copulas exist only
in the Python package - outside the parity scope.)


In [ ]:
# ---- setup: Python package + constants --------------------------------------
try:
    import genhmm1d
except ImportError:                       # e.g. on Google Colab
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/mamadouyamar/GenHMM1d.git"], check=True)

import numpy as np
from genhmm1d.hmm import HMM
hmm = HMM()

N        = 2000     # sample size
MAX_ITER = 10000
EPS      = 1e-6     # stopping criterion, passed to both sides
NINIT    = 100      # R hardcodes 100 minimum EM iterations; matched in Python
TOL      = 1e-2     # parity tolerance (optimizer internals differ slightly)
results  = {}

%load_ext rpy2.ipython


In [1]:
%%R
# ---- setup: R package (takes a few minutes on Colab) ------------------------
options(repos = "https://cloud.r-project.org")
if (!requireNamespace("GenHMM1d", quietly = TRUE)) install.packages("GenHMM1d")
suppressMessages(library(GenHMM1d))
cat("R GenHMM1d version:", as.character(packageVersion("GenHMM1d")), "\n")


R GenHMM1d version: 0.2.6 


## Gaussian, three regimes

In [1]:
%%R -i N,MAX_ITER,EPS -o y,th_R,Q_R
set.seed(101)

theta = matrix(c(0, 3, 6,
                 1, 1, 1),
               3, 2)              # regime means 0, 3, 6; sds 1

Q = matrix(c(0.90, 0.05, 0.05,
             0.05, 0.90, 0.05,
             0.05, 0.05, 0.90),
           3, 3)

sim = SimHMMGen(theta,
                Q = Q,
                family = "gaussian",
                n = N)

y = as.numeric(sim$SimData)

est = EstHMMGen(sim$SimData,
                ZI = 0,
                reg = 3,
                family = "gaussian",
                max_iter = MAX_ITER,
                eps = EPS)

th_R = est$theta; Q_R = est$Q


In [1]:
out = hmm.EstHMMGen(np.asarray(y).reshape(-1, 1), 3, 'norm',
                    max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=0)
th_P, Q_P = np.asarray(out["theta"], float), np.asarray(out["Q"], float)
th_R, Q_R = np.asarray(th_R, float).reshape(th_P.shape), np.asarray(Q_R, float).reshape(3, 3)

# align regime labels (R and Python may order the regimes differently)
o = np.argsort(th_R[:, 0]); th_R, Q_R = th_R[o], Q_R[np.ix_(o, o)]
o = np.argsort(th_P[:, 0]); th_P, Q_P = th_P[o], Q_P[np.ix_(o, o)]

theta_true = np.array([[0.0, 1.0], [3.0, 1.0], [6.0, 1.0]])
d_th, d_Q = np.max(np.abs(th_R - th_P)), np.max(np.abs(Q_R - Q_P))
print("theta_true:\n", theta_true)
print("theta_R   :\n", np.round(th_R, 4))
print("theta_Py  :\n", np.round(th_P, 4))
print("Q_R :\n", np.round(Q_R, 4))
print("Q_Py:\n", np.round(Q_P, 4))
print(f"max |R-Py|: theta {d_th:.2e}  Q {d_Q:.2e}  "
      f"[{'PASS' if max(d_th, d_Q) < TOL else 'FAIL'}]")
results["gaussian"] = (d_th, d_Q)


theta_true:
 [[0. 1.]
 [3. 1.]
 [6. 1.]]
theta_R   :
 [[1.1000e-03 9.6550e-01]
 [3.0160e+00 1.0624e+00]
 [6.0931e+00 1.0009e+00]]
theta_Py  :
 [[1.2000e-03 9.6560e-01]
 [3.0160e+00 1.0624e+00]
 [6.0931e+00 1.0008e+00]]
Q_R :
 [[0.8927 0.0426 0.0647]
 [0.0348 0.9273 0.0379]
 [0.0537 0.0675 0.8787]]
Q_Py:
 [[0.8927 0.0426 0.0647]
 [0.0348 0.9273 0.0379]
 [0.0537 0.0675 0.8787]]
max |R-Py|: theta 9.34e-05  Q 1.95e-06  [PASS]


## Poisson, three regimes

In [1]:
%%R -i N,MAX_ITER,EPS -o y,th_R,Q_R
set.seed(102)

theta = matrix(c(2, 9, 20),
               3, 1)              # lambda = 2, 9, 20

Q = matrix(c(0.90, 0.05, 0.05,
             0.05, 0.90, 0.05,
             0.05, 0.05, 0.90),
           3, 3)

sim = SimHMMGen(theta,
                Q = Q,
                family = "poisson",
                n = N)

y = as.numeric(sim$SimData)

est = EstHMMGen(sim$SimData,
                ZI = 0,
                reg = 3,
                family = "poisson",
                max_iter = MAX_ITER,
                eps = EPS)

th_R = est$theta; Q_R = est$Q


In [1]:
out = hmm.EstHMMGen(np.asarray(y).reshape(-1, 1), 3, 'poisson',
                    max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=0)
th_P, Q_P = np.asarray(out["theta"], float), np.asarray(out["Q"], float)
th_R, Q_R = np.asarray(th_R, float).reshape(th_P.shape), np.asarray(Q_R, float).reshape(3, 3)

# align regime labels (R and Python may order the regimes differently)
o = np.argsort(th_R[:, 0]); th_R, Q_R = th_R[o], Q_R[np.ix_(o, o)]
o = np.argsort(th_P[:, 0]); th_P, Q_P = th_P[o], Q_P[np.ix_(o, o)]

theta_true = np.array([[2.0], [9.0], [20.0]])
d_th, d_Q = np.max(np.abs(th_R - th_P)), np.max(np.abs(Q_R - Q_P))
print("theta_true:\n", theta_true)
print("theta_R   :\n", np.round(th_R, 4))
print("theta_Py  :\n", np.round(th_P, 4))
print("Q_R :\n", np.round(Q_R, 4))
print("Q_Py:\n", np.round(Q_P, 4))
print(f"max |R-Py|: theta {d_th:.2e}  Q {d_Q:.2e}  "
      f"[{'PASS' if max(d_th, d_Q) < TOL else 'FAIL'}]")
results["poisson"] = (d_th, d_Q)


theta_true:
 [[ 2.]
 [ 9.]
 [20.]]
theta_R   :
 [[ 2.0184]
 [ 8.8797]
 [19.7799]]
theta_Py  :
 [[ 2.0184]
 [ 8.8799]
 [19.7801]]
Q_R :
 [[0.8787 0.0721 0.0492]
 [0.0575 0.8906 0.0519]
 [0.0441 0.0395 0.9164]]
Q_Py:
 [[0.8787 0.0721 0.0492]
 [0.0575 0.8906 0.0519]
 [0.0441 0.0395 0.9164]]
max |R-Py|: theta 2.57e-04  Q 1.48e-06  [PASS]


## Zero-inflated Gaussian, three regimes

In [1]:
%%R -i N,MAX_ITER,EPS -o y,th_R,Q_R
set.seed(103)

theta = matrix(c(0, 3, 6,
                 0, 1, 1),
               3, 2)              # row 1 = point mass at 0

Q = matrix(c(0.90, 0.05, 0.05,
             0.05, 0.90, 0.05,
             0.05, 0.05, 0.90),
           3, 3)

sim = SimHMMGen(theta,
                Q = Q,
                ZI = 1,
                family = "gaussian",
                n = N)

y = as.numeric(sim$SimData)

est = EstHMMGen(sim$SimData,
                ZI = 1,
                reg = 3,
                family = "gaussian",
                max_iter = MAX_ITER,
                eps = EPS)

th_R = est$theta; Q_R = est$Q


In [1]:
out = hmm.EstHMMGen(np.asarray(y).reshape(-1, 1), 3, 'norm',
                    max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=1)
th_P, Q_P = np.asarray(out["theta"], float), np.asarray(out["Q"], float)
th_R, Q_R = np.asarray(th_R, float).reshape(th_P.shape), np.asarray(Q_R, float).reshape(3, 3)

# align regime labels (R and Python may order the regimes differently)
o = np.argsort(th_R[:, 0]); th_R, Q_R = th_R[o], Q_R[np.ix_(o, o)]
o = np.argsort(th_P[:, 0]); th_P, Q_P = th_P[o], Q_P[np.ix_(o, o)]

theta_true = np.array([[0.0, 0.0], [3.0, 1.0], [6.0, 1.0]])
d_th, d_Q = np.max(np.abs(th_R - th_P)), np.max(np.abs(Q_R - Q_P))
print("theta_true:\n", theta_true)
print("theta_R   :\n", np.round(th_R, 4))
print("theta_Py  :\n", np.round(th_P, 4))
print("Q_R :\n", np.round(Q_R, 4))
print("Q_Py:\n", np.round(Q_P, 4))
print(f"max |R-Py|: theta {d_th:.2e}  Q {d_Q:.2e}  "
      f"[{'PASS' if max(d_th, d_Q) < TOL else 'FAIL'}]")
results["zi-gaussian"] = (d_th, d_Q)


theta_true:
 [[0. 0.]
 [3. 1.]
 [6. 1.]]
theta_R   :
 [[0.     0.    ]
 [3.0469 0.9236]
 [6.0487 0.9762]]
theta_Py  :
 [[0.     0.    ]
 [3.0469 0.9236]
 [6.0487 0.9763]]
Q_R :
 [[0.9182 0.0371 0.0446]
 [0.064  0.8795 0.0565]
 [0.0457 0.0554 0.8989]]
Q_Py:
 [[0.9182 0.0371 0.0446]
 [0.064  0.8795 0.0565]
 [0.0457 0.0554 0.8989]]
max |R-Py|: theta 6.93e-05  Q 5.63e-07  [PASS]


## Zero-inflated Poisson, three regimes

In [1]:
%%R -i N,MAX_ITER,EPS -o y,th_R,Q_R
set.seed(104)

theta = matrix(c(0, 2, 9),
               3, 1)              # row 1 = point mass at 0

Q = matrix(c(0.90, 0.05, 0.05,
             0.05, 0.90, 0.05,
             0.05, 0.05, 0.90),
           3, 3)

sim = SimHMMGen(theta,
                Q = Q,
                ZI = 1,
                family = "poisson",
                n = N)

y = as.numeric(sim$SimData)

est = EstHMMGen(sim$SimData,
                ZI = 1,
                reg = 3,
                family = "poisson",
                max_iter = MAX_ITER,
                eps = EPS)

th_R = est$theta; Q_R = est$Q


In [1]:
out = hmm.EstHMMGen(np.asarray(y).reshape(-1, 1), 3, 'poisson',
                    max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=1)
th_P, Q_P = np.asarray(out["theta"], float), np.asarray(out["Q"], float)
th_R, Q_R = np.asarray(th_R, float).reshape(th_P.shape), np.asarray(Q_R, float).reshape(3, 3)

# align regime labels (R and Python may order the regimes differently)
o = np.argsort(th_R[:, 0]); th_R, Q_R = th_R[o], Q_R[np.ix_(o, o)]
o = np.argsort(th_P[:, 0]); th_P, Q_P = th_P[o], Q_P[np.ix_(o, o)]

theta_true = np.array([[0.0], [2.0], [9.0]])
d_th, d_Q = np.max(np.abs(th_R - th_P)), np.max(np.abs(Q_R - Q_P))
print("theta_true:\n", theta_true)
print("theta_R   :\n", np.round(th_R, 4))
print("theta_Py  :\n", np.round(th_P, 4))
print("Q_R :\n", np.round(Q_R, 4))
print("Q_Py:\n", np.round(Q_P, 4))
print(f"max |R-Py|: theta {d_th:.2e}  Q {d_Q:.2e}  "
      f"[{'PASS' if max(d_th, d_Q) < TOL else 'FAIL'}]")
results["zi-poisson"] = (d_th, d_Q)


theta_true:
 [[0.]
 [2.]
 [9.]]
theta_R   :
 [[0.    ]
 [2.2781]
 [8.8846]]
theta_Py  :
 [[0.   ]
 [1.964]
 [8.865]]
Q_R :
 [[0.8102 0.1312 0.0586]
 [0.1644 0.7887 0.047 ]
 [0.0522 0.0585 0.8894]]
Q_Py:
 [[0.9007 0.0404 0.0589]
 [0.0466 0.9035 0.0499]
 [0.0447 0.0668 0.8885]]
max |R-Py|: theta 3.14e-01  Q 1.18e-01  [FAIL]


## Parity summary

In [1]:
print(f"{'model':<14}{'max|dtheta|':>14}{'max|dQ|':>12}   verdict")
for k, (dt, dq) in results.items():
    print(f"{k:<14}{dt:>14.2e}{dq:>12.2e}   "
          f"{'PASS' if max(dt, dq) < TOL else 'FAIL'}")


model            max|dtheta|     max|dQ|   verdict
gaussian            9.34e-05    1.95e-06   PASS
poisson             2.57e-04    1.48e-06   PASS
zi-gaussian         6.93e-05    5.63e-07   PASS
zi-poisson          3.14e-01    1.18e-01   FAIL


### Reading the result

- **PASS everywhere**: on identical data the Python package reproduces the
  CRAN R package to numerical tolerance on every shared model class - the two
  implementations are the same estimator.
- **A FAIL is informative**: with identical data and deterministic
  initialization on both sides, a gap beyond optimizer tolerance points to a
  genuine difference between the code bases and identifies exactly which
  family and parameter to inspect.
